# 05 · IA generativa: personas a partir dos comentários
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/segmentacao-clientes-olist/blob/main/notebooks/05_ia_generativa_personas.ipynb)

**Ideia:** juntar uma amostra de comentários de avaliação (em português) de cada segmento e pedir a um LLM que resuma **o que esse grupo valoriza e do que reclama**, e proponha uma **persona** e uma mensagem de CRM.

**Chave de API:** guarde em *Secrets* do Colab (ícone de chave na lateral) com o nome `GEMINI_API_KEY` e ative o acesso deste notebook. **Nunca escreva a chave no notebook nem suba para o GitHub.**

**Dados e privacidade:** os comentários vêm da base pública da Olist, que já é anonimizada. Mesmo assim, enviamos ao modelo **só o texto** dos comentários e o perfil agregado do segmento, sem nenhum identificador de cliente ou pedido.

In [6]:
!pip install -q -U google-genai

In [7]:
import sqlite3, pandas as pd

try:                                    # no Colab: lê o banco do seu Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DADOS = '/content/drive/MyDrive/03_Portfolio/Projetos_ML/01_segmentacao-clientes-olist/dados'
except ModuleNotFoundError:             # fora do Colab: pasta dados/ do projeto
    DADOS = '../dados'

con = sqlite3.connect(f'{DADOS}/olist.db')

def q(sql):
    """Roda uma consulta SQL no banco da Olist e devolve um DataFrame."""
    return pd.read_sql(sql, con)


# ------------------------------------------------------------------------------
# Garante as tabelas rfm e segmentos (criadas nos notebooks 01 e 03).
# O Google Drive pode demorar a sincronizar o banco entre sessões do Colab; se elas
# não aparecerem aqui, são recriadas com exatamente o mesmo método do notebook 03.
# ------------------------------------------------------------------------------
tabelas = q("SELECT name FROM sqlite_master WHERE type='table'")['name'].tolist()
print('Banco em uso:', DADOS)
assert 'pedidos' in tabelas, f'O banco em {DADOS} está vazio: rode o notebook 00 antes.'

if 'segmentos' not in tabelas:
    print('Tabela segmentos não encontrada: recriando com o método do notebook 03...')
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    con.execute("""
    CREATE TABLE IF NOT EXISTS rfm AS
    WITH pago AS (SELECT order_id, SUM(payment_value) AS valor_pedido FROM pagamentos GROUP BY order_id),
    base AS (
        SELECT c.customer_unique_id, p.order_id, date(p.order_purchase_timestamp) AS data_compra, pg.valor_pedido
        FROM clientes c
        JOIN pedidos p ON c.customer_id = p.customer_id
        JOIN pago pg   ON p.order_id   = pg.order_id
        WHERE p.order_status = 'delivered'
    ),
    referencia AS (SELECT MAX(data_compra) AS ultima_data_base FROM base)
    SELECT b.customer_unique_id,
           CAST(julianday(r.ultima_data_base) - julianday(MAX(b.data_compra)) AS INT) AS recencia,
           COUNT(DISTINCT b.order_id) AS frequencia, ROUND(SUM(b.valor_pedido), 2) AS valor
    FROM base b CROSS JOIN referencia r
    GROUP BY b.customer_unique_id
    """)
    _rfm = q('SELECT * FROM rfm')
    _X = StandardScaler().fit_transform(pd.DataFrame({
        'recencia': _rfm['recencia'],
        'frequencia_log': np.log1p(_rfm['frequencia']),
        'valor_log': np.log1p(_rfm['valor'])}))
    _rfm['cluster'] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(_X)
    _p = _rfm.groupby('cluster')[['recencia', 'frequencia', 'valor']].mean()
    _nomes = {_p['frequencia'].idxmax(): 'Recorrentes'}
    _resto = _p.drop(list(_nomes))
    _nomes[_resto['recencia'].idxmax()] = 'Inativos'
    _resto = _resto.drop(list(_nomes), errors='ignore')
    _nomes[_resto['valor'].idxmax()] = 'Alto valor'
    _nomes[_resto['valor'].idxmin()] = 'Novos de baixo valor'
    _rfm['segmento'] = _rfm['cluster'].map(_nomes)
    _rfm[['customer_unique_id', 'recencia', 'frequencia', 'valor', 'segmento']].to_sql(
        'segmentos', con, if_exists='replace', index=False)
    con.commit()

print(q('SELECT segmento, COUNT(*) AS clientes FROM segmentos GROUP BY segmento ORDER BY segmento').to_string(index=False))

import textwrap

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Banco em uso: /content/drive/MyDrive/03_Portfolio/Projetos_ML/01_segmentacao-clientes-olist/dados
            segmento  clientes
          Alto valor     27832
            Inativos     27047
Novos de baixo valor     35677
         Recorrentes      2801


## 1. Amostra de comentários por segmento
`segmentos` × `clientes` × `pedidos` × `avaliacoes`, até 80 comentários não vazios por segmento (sorteio fixo, para dar sempre o mesmo resultado).

In [8]:
comentarios = q('''
SELECT s.segmento, a.review_score AS nota, a.review_comment_message AS comentario
FROM segmentos s
JOIN clientes c   ON s.customer_unique_id = c.customer_unique_id
JOIN pedidos p    ON c.customer_id = p.customer_id
JOIN avaliacoes a ON p.order_id = a.order_id
WHERE a.review_comment_message IS NOT NULL
  AND LENGTH(TRIM(a.review_comment_message)) >= 15     -- descarta comentários muito curtos
''')

amostra = (comentarios.sample(frac=1, random_state=42)    # embaralha com sorteio fixo
           .groupby('segmento').head(80))                 # e fica com até 80 por segmento

# Perfil agregado de cada segmento (vai junto no prompt)
perfil = q('''
SELECT segmento, COUNT(*) AS clientes,
       ROUND(AVG(recencia)) AS recencia_media, ROUND(AVG(frequencia), 2) AS frequencia_media,
       ROUND(AVG(valor), 2) AS valor_medio
FROM segmentos GROUP BY segmento
''').set_index('segmento')

display(perfil)
amostra.groupby('segmento').agg(comentarios=('comentario', 'count'), nota_media=('nota', 'mean')).round(2)

,clientes,recencia_media,frequencia_media,valor_medio
segmento,,,,
Alto valor,27832,173.0,1.00,318.36
Inativos,27047,425.0,1.00,119.57
Novos de baixo valor,35677,147.0,1.00,69.05
Recorrentes,2801,220.0,2.11,308.59


,comentarios,nota_media
segmento,,
Alto valor,80,3.59
Inativos,80,4.04
Novos de baixo valor,80,3.72
Recorrentes,80,3.65


## 2. Montar o prompt
Papel, contexto, formato de saída e a regra de **não inventar** o que não está nos comentários.

In [9]:
PROMPT = """Você é analista de CRM de um marketplace brasileiro.
Abaixo estão comentários reais de clientes do segmento "{segmento}" ({perfil}).
1. Liste os 3 temas mais elogiados e os 3 mais criticados, com a contagem aproximada.
2. Escreva uma persona de 3 linhas para esse segmento.
3. Sugira uma mensagem de CRM de até 200 caracteres.
Use apenas o que está nos comentários. Se algo não aparecer, diga que não há evidência.
Na base anonimizada, nomes de lojas foram trocados por nomes fictícios (ex.: "baratheon", "lannister"): trate-os como nomes de lojas.

Comentários (nota de 1 a 5 entre colchetes):
{comentarios}"""

def montar_prompt(segmento):
    p = perfil.loc[segmento]
    descricao = (f"{int(p['clientes'])} clientes; última compra há {int(p['recencia_media'])} dias em média; "
                 f"{p['frequencia_media']} pedidos e R$ {p['valor_medio']} gastos em média")
    linhas = amostra[amostra['segmento'] == segmento]
    texto = '\n'.join(f"- [{int(n)}] {c.strip()}" for n, c in zip(linhas['nota'], linhas['comentario']))
    return PROMPT.format(segmento=segmento, perfil=descricao, comentarios=texto)

print(montar_prompt('Alto valor')[:1500], '...')       # confira como o prompt ficou

Você é analista de CRM de um marketplace brasileiro.
Abaixo estão comentários reais de clientes do segmento "Alto valor" (27832 clientes; última compra há 173 dias em média; 1.0 pedidos e R$ 318.36 gastos em média).
1. Liste os 3 temas mais elogiados e os 3 mais criticados, com a contagem aproximada.
2. Escreva uma persona de 3 linhas para esse segmento.
3. Sugira uma mensagem de CRM de até 200 caracteres.
Use apenas o que está nos comentários. Se algo não aparecer, diga que não há evidência.
Na base anonimizada, nomes de lojas foram trocados por nomes fictícios (ex.: "baratheon", "lannister"): trate-os como nomes de lojas.

Comentários (nota de 1 a 5 entre colchetes):
- [1] produto veio errado
- [5] Produto entregue antes do prazo!
- [1] por enquanto, não recebi minha encomenda apesar de estar indicado no rastreio que ja foi entregue.
- [3] fui bem atendido, mas o valor cobrado foi divergente do valor apresentado no site(desconto de 5% pela forma de pagamento). mas tudo ja foi resolvi

## 3. Chamar o modelo e salvar as respostas
Usa o SDK oficial `google-genai`. O nome do modelo fica numa variável: se a Google trocar a versão, é só mudar aqui (lista em ai.google.dev/gemini-api/docs/models).

In [10]:
from google import genai
from google.colab import userdata

MODELO = 'gemini-3.5-flash-lite'                         # modelo rápido e barato para texto
cliente = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))   # a chave vem do Secrets, nunca do código

respostas = []
for segmento in ['Recorrentes', 'Alto valor', 'Novos de baixo valor', 'Inativos']:
    r = cliente.models.generate_content(model=MODELO, contents=montar_prompt(segmento))
    respostas.append({'segmento': segmento, 'resposta': r.text})
    print('=' * 80, f'\n{segmento}\n', '=' * 80)
    print(r.text)

respostas = pd.DataFrame(respostas)
respostas.to_sql('personas', con, if_exists='replace', index=False)   # guarda para o README e o painel
respostas.to_csv(f'{DADOS}/personas.csv', index=False)

Recorrentes
Com base exclusivamente nos comentários fornecidos, aqui está a análise:

### 1. Temas mais elogiados e mais criticados

**Temas mais elogiados:**
1. **Entrega rápida / antes do prazo:** ~19 menções (clientes destacando que o produto chegou antes do esperado, em menos de 7 dias ou com ótima eficiência).
2. **Qualidade e beleza do produto:** ~14 menções (elogios à boa qualidade, produtos bonitos, perfeitos, originais ou exatamente como anunciados).
3. **Bom atendimento / Embalagem / Acompanhamento:** ~9 menções (destaque para o cuidado com a embalagem, rastreio/comunicação transparente e atendimento das lojas).

**Temas mais criticados:**
1. **Não recebimento / Atraso na entrega:** ~14 menções (produtos não entregues, atrasos, problemas com os Correios e necessidade de buscar na agência).
2. **Problemas com itens faltantes ou incorretos (pedido errado):** ~11 menções (receber menos itens do que os comprados, cor, tamanho ou modelo diferentes do anunciado).
3. **Dificuldade c

## 4. Checar a resposta
Um LLM pode exagerar ou inventar. Leia 10 comentários de cada segmento e confira se os temas que o modelo apontou aparecem de fato.

In [11]:
for segmento, grupo in amostra.groupby('segmento'):
    print('=' * 80, f'\n{segmento}')
    for n, c in zip(grupo['nota'].head(10), grupo['comentario'].head(10)):
        print(f'  [{int(n)}]', textwrap.shorten(c, 160))

Alto valor
  [1] produto veio errado
  [5] Produto entregue antes do prazo!
  [1] por enquanto, não recebi minha encomenda apesar de estar indicado no rastreio que ja foi entregue.
  [3] fui bem atendido, mas o valor cobrado foi divergente do valor apresentado no site(desconto de 5% pela forma de pagamento). mas tudo ja foi resolvido.
  [5] Chegou antes do prazo previsto, cortina de ótima qualidade na minha opinião.
  [2] O produto chegou intacto, mas a embalagem toda rasgada e o produto parcialmente exposto.
  [5] Chegou bem antes do esperado 😍
  [5] Chegou relativamente rápido, antes do prazo previsto, produto lindo, condizente a foto, e veio em perfeito estado, muito bem embalado. Parabéns pelo [...]
  [5] Muito satisfeita. Produto entregue antes do prazo. Tudo em perfeita ordem.
  [5] Excelente vai ser muito útil. Muito obrigado!
Inativos
  [5] Gostei recebi tudo certinho no prazo estipulado.Parabéns para Loja.
  [5] Chegou bem antes do prazo! Parabéns!
  [4] Entregue antes da data

**Onde o modelo acertou e onde exagerou** (preencha depois de ler):
- **Acertou:** os temas batem com os comentários. **Entrega rápida / antes do prazo** é o elogio nº 1 em todos os segmentos; uma contagem simples por palavras-chave na mesma amostra encontrou de 15 a 29 comentários (de 80) por segmento, a mesma ordem de grandeza que o modelo indicou. As críticas principais (atraso ou não recebimento, itens faltantes ou errados) também aparecem na leitura dos comentários.
- **Exagerou ou errou:** (1) as **mensagens de CRM saíram quase iguais** entre os segmentos ("Sentimos sua falta!"), inclusive para Recorrentes e Novos, que não estão afastados; (2) chamou a compra do segmento **Alto valor** de "valor moderado", embora seja o maior ticket médio (R$ 318); (3) a persona dos Recorrentes cita lojas fictícias ("baratheon", "lannister") sem utilidade; (4) as contagens são aproximadas, e algumas parecem infladas (itens faltantes: o modelo diz ~9 a ~11, a contagem por palavras-chave encontrou de 2 a 4, embora o modelo some também produtos errados).

**Conclusão:** o LLM é útil para **resumir centenas de comentários em minutos** e confirmou, com a voz do cliente, o que os dados já mostravam: **a logística é o tema central da satisfação** (como na pergunta 6 do notebook 01). Mas as personas e mensagens **precisam de revisão humana**. Melhoria para a próxima versão: incluir no prompt o **objetivo de CRM de cada segmento** (2ª compra, aumento de ticket, reativação, fidelização), para que as mensagens deixem de ser genéricas.

## Gravar no Google Drive
Fecha o banco e força o Drive a gravar as mudanças agora. Sem isso, o próximo notebook, que roda em outra sessão do Colab, pode não enxergar as tabelas novas.

In [12]:
con.commit()
con.close()
try:
    drive.flush_and_unmount()      # só existe no Colab
    print('Banco gravado no Google Drive.')
except NameError:
    print('Fora do Colab: nada a sincronizar.')

Banco gravado no Google Drive.
